# 流式输出
通过`agent.stream(stream_mode=指定模式) `来指定。具体模式有：
- values
- updates（默认）
- messages
- custom
- checkpoints
- tasks
- debug

## 具体的输出模式
### values
当 stream_mode 设置为values模式时，每个步骤执行后，都会输出完整的状态信息，适用于每一步都
要获取完整状态、状态持久化场景。

In [ ]:
from langchain.agents import create_agent
from dotenv import load_dotenv
import os
from langchain_qwq import ChatQwen
from rich import print as rprint
from langchain.tools import tool

load_dotenv(override=True)
model = ChatQwen(
    model="qwen3.6-27b",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

In [ ]:
from typing import Dict, Any
@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}

@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}

@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }
# 创建客户服务Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)
for chunk in customer_service_agent.stream({"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},stream_mode="values"
):
    rprint(chunk)
    print("-" * 50)

### updates输出模式
这种模式就是默认模式。该模式中，每个步骤执行后，只增量更新状态中发生变化的内容，用于监控
Agent 执行进度，例如观察Agent决定调用工具、工具执行结果等步骤。

In [ ]:
for chunk in customer_service_agent.stream({"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},stream_mode="updates"
):
    rprint(chunk)
    print("-" * 50)

### messages输出模式
该模式中会输出流式返回的Token以及相关的元数据（如：来自哪个节点），可以用在实现类似 ChatGPT 的打字机效果场景，为聊天机器人等交互式应用提供最佳的实时体验。

In [ ]:
for chunk in customer_service_agent.stream({"messages":[{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},stream_mode="messages"):
    rprint(chunk)
    rprint("-" * 50)
# print(chunk[0].content,end="",flush=True)

###  tasks输出模式
该模式会输出当前task任务开始和结束的时间，包含任务的结果和错误信息，该模式用于监控任务的生命周期。

In [ ]:
for chunk in customer_service_agent.stream({"messages": [{"role": "user","content": "查询客户ID为  CUST123456 的完整信息和可用优惠"}]},stream_mode="tasks"):
    rprint(chunk)
    rprint("-" * 50)


### debug输出模式
该模式与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result），该模式用于调试、监控task任务的生命周期。

### checkpoints输出模式
该模式中，每当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态，用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# 其他工具代码同上，保持不变

# 1. 创建内存检查点存储
checkpointer = InMemorySaver()
# 2. 创建Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history,get_current_promotions],
    checkpointer=checkpointer # 启用检查点
)

# 3. 创建唯一的会话ID
config = {"configurable": {"thread_id": "session01"}}

# 4. 调用Agent
checkpoint_count = 0

# 使用checkpoints模式进行流式监控
for chunk in customer_service_agent.stream(
    {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
    config=config,
    stream_mode="checkpoints"
):
    checkpoint_count += 1
    rprint(f"检查点 #{checkpoint_count}")
    rprint(chunk)
    rprint("-" * 50)

### custom输出模式
通过 `get_stream_writer` 在工具或节点内部 `自定义发送的数据` ，用于 输出 业务逻辑相关的进度信息（如“已处理10/100条记录”）、自定义日志或指标。

In [9]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain.tools import tool
import time
@tool
def generate_sales_report() -> str:
    """生成销售报告"""
    writer = get_stream_writer()
    writer({"type": "生成销售报告", "message": "开始生成销售报告"})
    # 模拟数据处理
    for i in range(1, 4):
        time.sleep(0.5)
        writer({"type": "生成销售报告","message": f"生成销售报告进度百分比：{i * 25}%"})
    writer({"type": "生成销售报告", "message": "报告生成完成"})
    return f"销售报告：总收入150万元，同比增长12%"

@tool
def generate_inventory_report() -> str:
    """生成库存报告"""
    writer = get_stream_writer()
    writer("开始库存分析...")
    time.sleep(0.5)
    writer("检查当前库存量...")
    time.sleep(0.5)
    writer("生成库存报告...")
    return "当前库存量为10000件，库存充足，无异常"

# 创建报告生成agent
reporting_agent = create_agent(
    model=model,tools=[generate_sales_report, generate_inventory_report]
)
for chunk in reporting_agent.stream(
    {"messages": [{"role": "user","content": "生成销售报告和库存报告"}]},stream_mode="custom"
):
    print(chunk)
    print("-" * 50)

{'type': '生成销售报告', 'message': '开始生成销售报告'}
--------------------------------------------------
开始库存分析...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：25%'}
--------------------------------------------------
检查当前库存量...
--------------------------------------------------
生成库存报告...
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：50%'}
--------------------------------------------------
{'type': '生成销售报告', 'message': '生成销售报告进度百分比：75%'}
--------------------------------------------------
{'type': '生成销售报告', 'message': '报告生成完成'}
--------------------------------------------------


![Agent stream输出](../assets/agentStream.png)